# Convergence 3D elasticity to plates
We investigate numerically how good the Reissner-Mindlin plate and Kirchhoff-Love plate approximate the full 3D elasticity problem for thickness $t\to0$.

In [ ]:
from ngsolve import *
from ngsolve.krylovspace import CGSolver
from ngsolve.meshes import MakeStructured2DMesh, MakeStructured3DMesh
from ngsolve.webgui import Draw

We choose material parameters such that the full 3D elasticity tensor $\mathbb{C}$ becomes the identity. We consider a beam of length $1$ and thickness $t$, which is fixed on the left and apply a shear force at the right boundary.

In [ ]:
# use material parameters such that equations simplify
mu, lam = 0.5, 0
nu = lam / (2 * (lam + mu))
E = mu * (3 * lam + 2 * mu) / (lam + mu)
k = 5 / 6
G = E / (2 * (1 + nu))
force = CF((0, 0, -1))

mesh = MakeStructured3DMesh(nx=8, ny=8, nz=1, mapping=lambda x, y, z: (x, y, 0.1 * z))
Draw(mesh)

We use the [TDNNS method](../elasticity/TDNNS_elasticity.ipynb) for solving the 2D elasticity problem as it is robust with respect to the aspect ratio when using quadrilateral elements (see <a href="http://doi.org/10.1002/nme.3319">Pechstein and Schöberl. Anisotropic mixed finite elements for elasticity. <i>International Journal for Numerical Methods in Engineering </i>, 90(2)
  (2012), 196-217.</a>)

In [ ]:
def SolveTDNNS(order, mesh, t, draw=False):
    clamped = "left|right|back"
    # free = "front|top|bottom"
    fesU = HCurl(mesh, order=order, dirichlet=clamped)
    fesS = Discontinuous(HDivDiv(mesh, order=order))
    fesH = NormalFacetFESpace(mesh, order=order, dirichlet=clamped)
    X = fesS * fesU * fesH
    (sigma, u, uh), (tau, v, vh) = X.TnT()

    n = specialcf.normal(3)

    a = BilinearForm(X, symmetric=True, symmetric_storage=True, condense=True)
    a += (
        InnerProduct(sigma, tau)
        - InnerProduct(sigma, Grad(v))
        - InnerProduct(tau, Grad(u))
    ) * dx
    a += (sigma[n, n] * (v - vh) * n + tau[n, n] * (u - uh) * n) * dx(
        element_boundary=True
    )
    a.Assemble()

    f = LinearForm(X)
    f += -(t**2) * force * v * dx
    f.Assemble()

    gf_sol = GridFunction(X)
    _, gf_u, _ = gf_sol.components

    invS = a.mat.Inverse(X.FreeDofs(a.condense), inverse="sparsecholesky")
    if a.condense:
        f.vec.data += a.harmonic_extension_trans * f.vec
    gf_sol.vec.data = invS * f.vec
    if a.condense:
        gf_sol.vec.data += a.harmonic_extension * gf_sol.vec
        gf_sol.vec.data += a.inner_solve * f.vec

    if draw:
        Draw(BoundaryFromVolumeCF(gf_u), mesh, "u", deformation=True)

    return gf_u  # (mesh(0.5, 1, 0))[2]


t = 1e-2
mesh = MakeStructured3DMesh(
    nx=12, ny=12, nz=1, mapping=lambda x, y, z: (x, y, t * (z - 0.5))
)

SolveTDNNS(
    2,
    t=t,
    mesh=mesh,
    draw=True,
);


The [Reissner-Mindlin plate](Reissner_Mindlin_plate.ipynb) via TDNNS and [Kirchhoff-Love plate](Kirchhoff_Love_plate.ipynb) via HHJ.

In [ ]:
t = 1e-2


def CMatInv(mat, E, nu):
    return (
        12
        * (1 - nu**2)
        / E
        * (1 / (1 - nu) * mat - nu / (1 - nu**2) * Trace(mat) * Id(2))
    )


def SolveRM_TDNNS(mesh, t, order=1, draw=False):
    fesB = HCurl(mesh, order=order - 1, dirichlet="left|top|bottom")
    fesS = HDivDiv(mesh, order=order - 1, dirichlet="right")
    fesW = H1(mesh, order=order, dirichlet="left|top|bottom")

    fes = fesW * fesB * fesS
    (w, beta, sigma), (v, delta, tau) = fes.TnT()

    n = specialcf.normal(2)

    a = BilinearForm(fes, symmetric=True)
    a += (
        -InnerProduct(CMatInv(sigma, E, nu), tau)
        + InnerProduct(tau, grad(beta))
        + InnerProduct(sigma, grad(delta))
    ) * dx
    a += (-(sigma * n * n) * (delta * n) - (tau * n * n) * (beta * n)) * dx(
        element_boundary=True
    )
    a += k * G / t**2 * InnerProduct(grad(w) - beta, grad(v) - delta) * dx

    f = LinearForm(fes)
    f += force[2] * v * dx

    gfsol = GridFunction(fes, autoupdate=True)
    gfw, gfbeta, _ = gfsol.components

    with TaskManager():
        a.Assemble()
        f.Assemble()
        inv = a.mat.Inverse(fes.FreeDofs(), inverse="umfpack")
        gfsol.vec.data = inv * f.vec
    if draw:
        Draw(gfw, mesh, "w", deformation=True)

    return gfw, gfbeta  # (mesh(1, 0.5))


SolveRM_TDNNS(t=t, order=2, mesh=MakeStructured2DMesh(nx=10, ny=10), draw=True)


def SolveKL_HHJ(mesh, order=2, draw=False):
    V = HDivDiv(mesh, order=order - 1, dirichlet="right")
    Q = H1(mesh, order=order, dirichlet="left|bottom|top")
    X = V * Q
    (sigma, w), (tau, v) = X.TnT()

    n = specialcf.normal(2)

    def tang(u):
        return u - (u * n) * n

    a = BilinearForm(X, symmetric=True)
    a += (
        -InnerProduct(CMatInv(sigma, E, nu), tau)
        - div(sigma) * grad(v)
        - div(tau) * grad(w)
    ) * dx - (-(sigma * n) * tang(grad(v)) - (tau * n) * tang(grad(w))) * dx(
        element_boundary=True
    )

    f = LinearForm(X)
    f += force[2] * v * dx

    gfsol = GridFunction(X)
    _, gfw = gfsol.components
    with TaskManager():
        a.Assemble()
        f.Assemble()
        gfsol.vec.data = a.mat.Inverse(X.FreeDofs(), inverse="umfpack") * f.vec

    if draw:
        Draw(gfw, mesh, name="w", deformation=True)

    return gfw  # (mesh(1, 0.5))


SolveKL_HHJ(order=2, mesh=MakeStructured2DMesh(nx=10, ny=10), draw=True);

Solve the 3D elasticity and Reissner-Mindlin plate problem for different thicknesses $t$ (the Kirchhoff-Love plate equation is independent of $t$ and gets thus only solved once) and compute the relative error of the plate models with respect to the 3D elasticity solution by comparing the vertical deflection at the midsurface at the right boundary at the middle.

In [ ]:
l = []

# with TaskManager():
print("A")
gfw_KL = SolveKL_HHJ(order=3, mesh=MakeStructured2DMesh(nx=30, ny=30))
gfu_KL = CF((-z * Grad(gfw_KL), gfw_KL))
print("B")
for i, t in enumerate([10 ** (-i) for i in range(4)]):
    print("t=", t)
    mesh3D = MakeStructured3DMesh(
        nx=8 * (i + 1),
        ny=8 * (i + 1),
        nz=2,
        mapping=lambda x, y, z: (x, y, t * (z - 0.5)),
    )
    print("C")
    gfuTDNNS = SolveTDNNS(
        order=2,
        t=t,
        mesh=mesh3D,
    )
    l2TDNNS = sqrt(Integrate(gfuTDNNS * gfuTDNNS, mesh3D))
    gfw_RM, gfbeta_RM = SolveRM_TDNNS(
        order=3, mesh=MakeStructured2DMesh(nx=30, ny=30), t=t
    )
    gfu_RM = CF((-z * gfbeta_RM, gfw_RM))
    # l.append(
    #     (
    #         t,
    #         abs(displ_RM - displ_el) / abs(displ_el),
    #         abs(displ_KL - displ_el) / abs(displ_el),
    #     )
    # )
    l.append(
        (
            t,
            sqrt(Integrate((gfuTDNNS - gfu_RM) ** 2, mesh3D)) / l2TDNNS,
            sqrt(Integrate((gfuTDNNS - gfu_KL) ** 2, mesh3D)) / l2TDNNS,
        )
    )
    print("l = ", l[-1])

In [ ]:
import matplotlib.pyplot as plt

plt.yscale("log")
plt.xscale("log")
plt.xlabel("t")
plt.ylabel("relative err")
ts, errrm, errkl = zip(*l)
plt.plot(ts, errrm, "*-", label="|u_{RM}-u_{3D}|$")
plt.plot(ts, errkl, "x-", label="|u_{KL}-u_{3D}|$")
plt.plot(ts, [th**2 for th in ts], "-", color="k", label="$O(t^2)$")
plt.plot(ts, [th**3 for th in ts], "-.", color="k", label="$O(t^3)$")
plt.legend()
plt.show()

We observe convergence, however, the point of rounding errors due to the small thickness is rapidly reached. Next, we compare the convergence of the Reissner-Mindlin plate to the Kirchhoff-Love plate model for $t\to0$. Here the point of loss of convergence is later at about $10^{-8}$ relative error (half of double precision).

In [ ]:
l = []
displ_KL = SolveKL_HHJ(order=3, mesh=MakeStructured2DMesh(nx=20, ny=20))
with TaskManager():
    for t in [10 ** (-i) for i in range(7)]:
        print("t=", t)
        displ_RM = SolveRM_TDNNS(order=3, mesh=MakeStructured2DMesh(nx=20, ny=20), t=t)
        l.append((t, abs(displ_KL - displ_RM) / abs(displ_KL)))

import matplotlib.pyplot as plt

plt.yscale("log")
plt.xscale("log")
plt.xlabel("t")
plt.ylabel("relative err")
ts, err = zip(*l)
plt.plot(ts, err, "*-", label="|u_{RM}-u_{KL}|$")
plt.plot(ts, [th**2 for th in ts], "-", color="k", label="$O(t^2)$")
plt.legend()
plt.show()

By introducing the shear $\gamma=\frac{\kappa\,G}{t^2}(\nabla w-\beta)$ we can reformulate the (TDNNS) Reissner-Mindlin plate equation as mixed saddle-point problem. Find $(w,\beta,\sigma,\gamma)$ such that for all $(v,\delta,\tau,d\gamma)$
\begin{align}
\begin{array}{llllll}
&-\int_{\Omega} \mathbb{C}^{-1} \sigma : \tau \,dx &-  \left< \mathrm{div} \tau, \beta \right> && &= & 0 \\
&-\left< \mathrm{div} \sigma, \delta \right>  &&+   \int_{\Omega} \gamma\cdot (\nabla v - \delta)\,dx & &= & \int_{\Omega} f v\,dx\\
&&\int_{\Omega}(\nabla w - \beta)\cdot d\gamma\,dx &&- \frac{t^2}{\kappa\, G}\int_{\Omega}\gamma\cdot d\gamma\,dx &=&0
\end{array}
\end{align}


In [ ]:
def SolveRM_TDNNS_Mixed(mesh, t, order=1, draw=False):
    fesB = HCurl(mesh, order=order - 1, dirichlet="left|top|bottom")
    fesS = HDivDiv(mesh, order=order - 1, dirichlet="right")
    fesW = H1(mesh, order=order, dirichlet="left|top|bottom")

    fes = fesW * fesB * fesS * fesB
    (w, beta, sigma, gamma), (v, delta, tau, dgamma) = fes.TnT()

    n = specialcf.normal(2)

    a = BilinearForm(fes, symmetric=True)
    a += (
        -InnerProduct(CMatInv(sigma, E, nu), tau)
        + InnerProduct(tau, grad(beta))
        + InnerProduct(sigma, grad(delta))
    ) * dx
    a += (-(sigma * n * n) * (delta * n) - (tau * n * n) * (beta * n)) * dx(
        element_boundary=True
    )
    a += (
        InnerProduct(gamma, grad(v) - delta)
        + InnerProduct(grad(w) - beta, dgamma)
        - t**2 / (k * G) * gamma * dgamma
    ) * dx

    f = LinearForm(fes)
    f += force[2] * v * dx

    gfsol = GridFunction(fes, autoupdate=True)
    gfw, gfbeta, gfsigma, gfgamma = gfsol.components

    with TaskManager():
        a.Assemble()
        f.Assemble()
        inv = a.mat.Inverse(fes.FreeDofs(), inverse="umfpack")
        gfsol.vec.data = inv * f.vec
    if draw:
        Draw(gfw, mesh, "w", deformation=True)

    return gfw(mesh(1, 0.5))


print(
    "RM mixed: ",
    SolveRM_TDNNS_Mixed(
        t=t, order=2, mesh=MakeStructured2DMesh(nx=10, ny=10), draw=True
    ),
)


In [ ]:
l = []
displ_KL = SolveKL_HHJ(order=3, mesh=MakeStructured2DMesh(nx=20, ny=20))
with TaskManager():
    for t in [10 ** (-i) for i in range(7)]:
        print("t=", t)
        displ_RM = SolveRM_TDNNS_Mixed(
            order=3, mesh=MakeStructured2DMesh(nx=20, ny=20), t=t
        )
        l.append((t, abs(displ_KL - displ_RM) / abs(displ_KL)))

import matplotlib.pyplot as plt

plt.yscale("log")
plt.xscale("log")
plt.xlabel("t")
plt.ylabel("relative err")
ts, err = zip(*l)
plt.plot(ts, err, "*-", label="|u_{RM}-u_{KL}|$")
plt.plot(ts, [th**2 for th in ts], "-", color="k", label="$O(t^2)$")
plt.legend()
plt.show()